# PlanDと原資料の一致度監査

2026-10-25。図の再現性と実被害への予測精度を区別する。全件監査は別実装のXML読取り・属性分類を用いるが、原図の照合は同じ軸校正と色を共有する。マーカーの重なりによる読取り偏りも残る。実被害の正解ラベルは結合していない。

詳しい解釈・出典は `docs/superpowers/plans/2026-10-25-building-accuracy-and-eew-research.md`。

In [1]:
from pathlib import Path
import json
import csv
import sys
import subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'tools/building_analysis').is_dir())
data_dir = root/'docs/superpowers/plans/data'
report = json.loads((data_dir/'2026-10-25-building-agreement.json').read_text(encoding='utf-8'))
print('Model:', report['modelId'])
print('Audit:', report['auditDate'])

Model: pland-cabinet-2025-digitized-v1
Audit: 2026-10-25


## 元資料からの再実行（任意）

初期値False。Trueへ変更すると保存済みのPDFと約6GBのCityGMLを読み直す。原資料の取得方法は `tools/building_analysis/README.md`。数分程度かかる。製品資産は変更しない。

In [2]:
RUN_FULL_AUDIT = False
if RUN_FULL_AUDIT:
    output = root/'.implementation/building-agreement-rerun'
    subprocess.run([sys.executable, str(root/'tools/building_analysis/audit_reference_agreement.py'),
        '--assets', str(root/'android/app/src/main/assets/kyoto-building-assessment'),
        '--pdf', str(root/'.implementation/nankai-2025-method.pdf'),
        '--cache', str(root/'.implementation/kyoto'), '--out', str(output)], check=True)
    report = json.loads((output/'agreement.json').read_text(encoding='utf-8'))
else:
    print('全件再実行は省略。保存済みの全件監査結果を以下で確認する。')

全件再実行は省略。保存済みの全件監査結果を以下で確認する。


## データ対応とカバー率

母数は公開建物モデル数。現地確認した建物数や正解ラベル数ではない。

In [3]:
counts = report['source']['counts']
assert counts['source_buildings'] == counts['matched_buildings'] == 513815
assert counts['meshes'] == 242
assert not report['source']['issues']
assert set(counts) == {'source_buildings', 'matched_buildings', 'meshes'}, counts
assert report['copied_curve_nodes'] == {'equal': 882, 'total': 882}
profiles = report['source']['profiles']
assert sum(profiles.values()) == counts['source_buildings']
unknown = profiles['unknown']; assumed = profiles['highrise-zero']
estimated = counts['source_buildings'] - unknown - assumed
for label, value in [('estimated', estimated), ('unknown', unknown), ('assumed', assumed)]:
    print(f'{label}: {value:,} / {counts["source_buildings"]:,} ({value/counts["source_buildings"]*100:.4f}%)')
print('Source/attribute/classification issues:', len(report['source']['issues']))
print('Copied values:', report['copied_curve_nodes'])

estimated: 393,152 / 513,815 (76.5163%)
unknown: 120,648 / 513,815 (23.4808%)
assumed: 15 / 513,815 (0.0029%)
Source/attribute/classification issues: 0
Copied values: {'equal': 882, 'total': 882}


## 原図の診断値を点別CSVから再集計

6.0〜7.0に限定。除外点を成功として数えない。これは信頼区間・被害予測誤差ではない。グラフの色・軸は共通で、部分的なマーカー隠れによる偏りも含む。

In [4]:
import numpy as np
points_path = output/'raster-points.csv' if RUN_FULL_AUDIT else data_dir/'2026-10-25-raster-points.csv'
with points_path.open(encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
assert len(rows) == 882
for kind, summary_key in [('node','nodes'), ('midpoint','midpoints')]:
    subset = [r for r in rows if r['point_kind'] == kind]
    good = [r for r in subset if r['status'] == 'compared']
    error = np.array([float(r['absolute_difference_pp']) for r in good])
    expected = report['raster'][summary_key]
    assert len(good) == expected['compared']
    assert abs(error.mean()-expected['mean_absolute_difference_pp']) < 1e-10
    assert abs(error.max()-expected['max_absolute_difference_pp']) < 1e-10
    print(kind, 'compared', len(good), '/', len(subset), 'mean', round(error.mean(),4),
          'p95', round(float(np.quantile(error,.95)),4), 'max', round(error.max(),4), 'percentage points')
print('実被害の予測精度は未測定。')

node compared 443 / 462 mean 0.3949 p95 1.0233 max 2.0466 percentage points
midpoint compared 401 / 420 mean 0.2568 p95 0.6277 max 0.9068 percentage points
実被害の予測精度は未測定。


## 入力震度への感度

同一の保存済み被害曲線で入力が0.1異なる場合の変化量。実際の予測誤差を測ったものではない。

In [5]:
curves = json.loads((root/'tools/building_analysis/curves-2025.json').read_text(encoding='utf-8'))['curves']
changes = [(b-a, profile, metric, round(5+i/10,1), a, b)
           for profile, pair in curves.items() for metric, values in pair.items()
           for i, (a,b) in enumerate(zip(values, values[1:]))]
print('Largest changes for intensity +0.1:', sorted(changes, reverse=True)[:6])
print('w0 full at 6.1, 6.2:', curves['w0']['full'][11:13])

Largest changes for intensity +0.1: [(14, 'w0', 'full', 6.3, 57, 71), (14, 'w0', 'full', 6.2, 43, 57), (14, 'w0', 'full', 6.1, 29, 43), (14, 'r1l', 'fullHalf', 6.9, 59, 73), (14, 'r1l', 'fullHalf', 6.8, 45, 59), (14, 'r1h', 'fullHalf', 6.5, 53, 67)]
w0 full at 6.1, 6.2: [29, 43]
